## Simple train out and back example

In [1]:
import asyncio
import logging
from pyjmri import Client

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
    filename="pyjmri.log",
    force=True,
)

## Connect to my layout

In [5]:
jmri = await Client("192.168.1.159:12080").__aenter__()
layout = await jmri.discover()
print(f"discovered: {len(layout.turnouts)} turnouts, {len(layout.sensors)} sensors, {len(layout.routes)} routes, {len(layout.blocks)} blocks")

discovered: 52 turnouts, 63 sensors, 15 routes, 38 blocks


## Some helper functions

In [ ]:
async def horn(t, *, duration: float = 1.2, pause: float = 0.5, times: int = 3) -> None:
    """Toggle F2 to blow the horn `times` times."""
    for _ in range(times):
        await t.set_function(2, True)
        await asyncio.sleep(duration)
        await t.set_function(2, False)
        await asyncio.sleep(pause)


async def wait_edge(sensor) -> None:
    """Wait for the sensor to go ACTIVE, then back to INACTIVE.

    The trailing edge means the loco has fully passed the detection
    block — used to know the train has cleared a turnout, not just
    reached it.
    """
    print(f"Waiting for {sensor} to be active")
    await sensor.wait_active()
    print(f"{sensor} is active")
    print(f"Waiting for {sensor} to become inactive")
    await sensor.wait_inactive()
    print(f"{sensor} is now inactive")



async def slow_through(layout, t, schedule: list[tuple[str, float]]) -> None:
    """At each (sensor_user_name, speed) checkpoint, wait for the sensor
    to go ACTIVE, then drop the throttle to the listed speed."""
    for sensor_name, speed in schedule:
        await layout.sensors[sensor_name].wait_active()
        await t.set_speed(speed, forward=True)

In [7]:
async def run_train(layout, dcc, staging_track:str, return_track:str, come_home: asyncio.Event) -> None:
    print(f"[{dcc}] preparing to depart {staging_track}")
    throat = layout.sensors["West / SW"]
    await layout.routes[staging_track].activate()


    async with layout.throttle(dcc, long=True) as t:
        await t.set_function(0, True)
        await t.set_speed(0.05, forward=True)
        print(f"[{dcc}] departing {staging_track}")
        await t.set_speed(0.2)
        await wait_edge(throat) 
        await layout.routes["NW Staging Close"].activate()
        await t.set_speed(0.4)
        # So now we are running and the mainline has been restored
        print("Now we fire the come home event, and we will fall into the return routine")
        await come_home.wait()
        print(f"{dcc} has been commanded to return to staging")
        await slow_through(layout, t, [
            ("North Zone 9", 0.20),
            ("West / SW", 0.15),
        ])
        await throat.wait_inactive()
        await t.set_speed(0)
        print("Train stopping, ready to reverse")
        await asyncio.sleep(10)
        await layout.routes[return_track].activate()
        # Start the bell for reverse direction
        await t.set_function(1, True)
        await t.set_speed(0.15, forward=False)
        await wait_edge(throat)
        await t.set_speed(0.10, forward=False)
        await asyncio.sleep(28)
        await t.set_speed(0)
        await t.set_function(1, False)  #bell off
        await t.set_function(0, False)  #light off
        await layout.routes["NW Staging Close"].activate()  #Restore main line, close NW
        print(f"{dcc} returned and parked in {return_track}")
        


## Now let's run a train with engine 8997

In [ ]:
come_home = asyncio.Event()
run_8997 = asyncio.create_task(run_train(layout, 8997, "NW Track 6", "NW Track 6", come_home))
print("Task created to launch the train")
await run_8997

## Close the client session

In [ ]:
await jmri.__aexit__(None, None, None)
print("CLient closed")